In [1]:
import pandas as pd

In [30]:
import numpy as np

In [4]:
# 1. Get current components
url = "https://en.wikipedia.org/wiki/NASDAQ-100"

# Create a dictionary with a browser-like User-Agent
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
}
# Pass the headers via storage_options
tables = pd.read_html(url, storage_options=headers)

In [9]:
len(tables)

20

In [15]:
df_current = tables[5]
df_changes = tables[6]

In [16]:
df_changes.columns = ['_'.join(col).strip() for col in df_changes.columns]
# Rename columns to friendly names based on your screenshot's structure
df_changes = df_changes.rename(columns={
    'Date_Date': 'Date',
    'Added_Ticker': 'Added_Ticker',
    'Added_Security': 'Added_Security',
    'Removed_Ticker': 'Removed_Ticker',
    'Removed_Security': 'Removed_Security'
})

In [17]:
# --- Step 2: Convert Dates and Filter ---
# Convert strings like "June 22, 2026" to datetime objects
df_changes['Date'] = pd.to_datetime(df_changes['Date'])

target_date = pd.to_datetime('2021-12-31')

# Filter for changes that happened AFTER our target date up to today
# We sort them from newest to oldest to walk backward correctly
changes_to_apply = df_changes[df_changes['Date'] > target_date].sort_values(by='Date', ascending=False)

In [42]:
# --- Step 3: Initialize with Current State ---
# Extract your current pool of tickers into a python set
# (Adjust 'Ticker' to whatever column name is in your current constituents table)
current_tickers = set(df_current['Ticker'].dropna().tolist())

# --- Step 4: Walk Backward ---
for _, row in changes_to_apply.iterrows():
    added = row['Added_Ticker']
    removed = row['Removed_Ticker']
    
    # Reverse the action:
    # If it was added recently, it wasn't there in 2022 -> Remove it
    if pd.notna(added) and added in current_tickers:
        current_tickers.remove(added)
        
    # If it was removed recently, it was still there in 2022 -> Add it back
    if pd.notna(removed):
        current_tickers.add(removed)

# --- Step 5: Output ---
tickers_2022 = sorted(list(current_tickers))
print(f"Total components in late 2022: {len(tickers_2022)}")
print(tickers_2022)

Total components in late 2022: 101
['AAPL', 'ABNB', 'ADBE', 'ADI', 'ADP', 'ADSK', 'AEP', 'ALGN', 'AMAT', 'AMD', 'AMGN', 'AMZN', 'ANSS', 'ASML', 'ATVI', 'AVGO', 'BIDU', 'BIIB', 'BKNG', 'CDNS', 'CHTR', 'CMCSA', 'COST', 'CPRT', 'CRWD', 'CSCO', 'CSX', 'CTAS', 'CTSH', 'DDOG', 'DLTR', 'DOCU', 'DXCM', 'EA', 'EBAY', 'EXC', 'FAST', 'FI', 'FTNT', 'GILD', 'GOOG', 'GOOGL', 'HON', 'IDXX', 'ILMN', 'INTC', 'INTU', 'ISRG', 'JD', 'KDP', 'KHC', 'KLAC', 'LCID', 'LRCX', 'LULU', 'MAR', 'MCHP', 'MDLZ', 'MELI', 'META', 'MNST', 'MRNA', 'MRVL', 'MSFT', 'MTCH', 'MU', 'NFLX', 'NTES', 'NVDA', 'NXPI', 'OKTA', 'ORLY', 'PANW', 'PAYX', 'PCAR', 'PDD', 'PEP', 'PTON', 'PYPL', 'QCOM', 'REGN', 'ROST', 'SBUX', 'SGEN', 'SIRI', 'SNPS', 'SPLK', 'SWKS', 'TEAM', 'TMUS', 'TSLA', 'TXN', 'VRSK', 'VRSN', 'VRTX', 'WBA', 'WDAY', 'XEL', 'XLNX', 'ZM', 'ZS']


In [23]:
df_old=df_current.merge(changes_to_apply[['Added_Ticker','Removed_Ticker']],left_on='Ticker',right_on='Added_Ticker',how='left')

In [24]:
df_old.shape

(101, 6)

In [35]:
old_tickers=np.where(df_old['Removed_Ticker'].isna(),df_old['Ticker'],df_old['Removed_Ticker'])

In [38]:
len(old_tickers)

101

In [61]:
','.join(list(set(tickers_2022).union(set(old_tickers))))

'ATVI,SNPS,CPRT,SBUX,SPLK,AMZN,AVGO,NFLX,KLAC,ABNB,JD,ILMN,INTU,FTNT,AMAT,ENPH,KHC,CSGP,AMD,DOCU,ANSS,ROST,MNST,TMUS,TRI,PTON,MDB,ADSK,MU,CDNS,LRCX,BKNG,CEG,NVDA,META,PEP,EXC,FAST,VRSN,INSM,ZM,COST,CTSH,ON,TEAM,MAR,NTES,INTC,PANW,VRTX,FI,PDD,AZN,MRVL,CTAS,GOOG,XEL,CSX,TXN,GOOGL,ORLY,VRSK,ASML,ZS,DDOG,TTD,ALGN,TSLA,AMGN,ADP,SMCI,NXPI,CSCO,BIDU,MELI,MTCH,ADBE,HON,MDLZ,EBAY,CRWD,LULU,PYPL,WDAY,REGN,DXCM,MCHP,AAPL,GILD,CHTR,SWKS,CMCSA,OKTA,LCID,SIRI,WBA,KDP,IDXX,XLNX,CDW,QCOM,AEP,PCAR,DLTR,ISRG,EA,MRNA,BIIB,PAYX,MSFT,SGEN,ADI'

In [41]:
old_tickers

array(['ADBE', 'AMD', 'ABNB', 'BIIB', 'GOOGL', 'GOOG', 'AMZN', 'AEP',
       'AMGN', 'ADI', 'AAPL', 'AMAT', 'DLTR', 'SIRI', 'ASML', 'CHTR',
       'ADSK', 'ADP', 'SMCI', 'MTCH', 'BKNG', 'AVGO', 'CDNS', 'CTAS',
       'CSCO', 'EBAY', 'CMCSA', 'CEG', 'CPRT', 'CTSH', 'COST', 'CRWD',
       'CSX', 'DDOG', 'DXCM', 'DOCU', 'ENPH', 'EA', 'EXC', 'FAST', 'CDW',
       'FTNT', 'FI', 'GILD', 'HON', 'IDXX', 'INTC', 'INTU', 'ISRG', 'KDP',
       'KLAC', 'KHC', 'LRCX', 'SPLK', 'CSGP', 'MAR', 'MRVL', 'MELI',
       'META', 'MCHP', 'MU', 'MSFT', 'MRNA', 'MDLZ', 'LULU', 'MNST',
       'INSM', 'NFLX', 'NVDA', 'NXPI', 'ORLY', 'PTON', 'PCAR', 'ILMN',
       'PANW', 'PAYX', 'PYPL', 'PDD', 'PEP', 'QCOM', 'REGN', 'VRSK',
       'LCID', 'ROST', 'TEAM', 'ON', 'MDB', 'SBUX', 'SNPS', 'TMUS',
       'SGEN', 'ZS', 'TSLA', 'TXN', 'TRI', 'VRTX', 'AZN', 'SPLK', 'TTD',
       'WDAY', 'XEL'], dtype=object)

# take samples

In [55]:
import random

def get_random_sample(original_list: list, sample_size: int) -> list:
    """
    Returns a sample of strictly unique elements from the original list,
    ensuring the returned list length equals sample_size.
    """
    # 1. Deduplicate the original list using a set, then convert back to a list
    unique_base_list = list(set(original_list))
    
    # 2. Check the guard rail against the *unique* count, not the original count
    if sample_size > len(unique_base_list):
        raise ValueError(
            f"Sample size ({sample_size}) cannot be larger than the number of "
            f"unique elements in the original list ({len(unique_base_list)})."
        )
        
    # 3. random.sample guarantees unique selection and exact length
    return random.sample(unique_base_list, sample_size)

In [58]:
sample_tickers=get_random_sample(old_tickers.tolist(),20)

In [59]:
sample_tickers

['ORLY',
 'VRTX',
 'PANW',
 'LRCX',
 'AAPL',
 'ADSK',
 'CEG',
 'MDB',
 'CSCO',
 'DDOG',
 'CTSH',
 'MELI',
 'SGEN',
 'IDXX',
 'TMUS',
 'AVGO',
 'ILMN',
 'LCID',
 'ISRG',
 'AEP']

# current tickers

In [65]:
df_current.columns

Index(['Ticker', 'Company', 'ICB Industry[15]', 'ICB Subsector[15]'], dtype='str')

In [68]:
df_current.shape

(101, 4)

In [67]:
'|'.join(df_current['Ticker'].tolist())

'ADBE|AMD|ABNB|ALNY|GOOGL|GOOG|AMZN|AEP|AMGN|ADI|AAPL|AMAT|APP|ARM|ASML|ALAB|ADSK|ADP|AXON|BKR|BKNG|AVGO|CDNS|CTAS|CSCO|CCEP|CMCSA|CEG|CPRT|CRWV|COST|CRWD|CSX|DDOG|DXCM|FANG|DASH|EA|EXC|FAST|FER|FTNT|GEHC|GILD|HON|IDXX|INTC|INTU|ISRG|KDP|KLAC|KHC|LRCX|LIN|LITE|MAR|MRVL|MELI|META|MCHP|MU|MSFT|MSTR|MDLZ|MPWR|MNST|NBIS|NFLX|NVDA|NXPI|ORLY|ODFL|PCAR|PLTR|PANW|PAYX|PYPL|PDD|PEP|QCOM|REGN|RKLB|ROP|ROST|SNDK|STX|SHOP|SBUX|SNPS|TMUS|TTWO|TER|TSLA|TXN|TRI|VRTX|WMT|WBD|WDC|WDAY|XEL'